# World air quality index analytics

The data base is available at https://www.kaggle.com/datasets/adityaramachandran27/world-air-quality-index-by-city-and-coordinates/data

The dataset contains the Air Quality Index (hereafter AQI) for various cities around the world, along with their coordinates, as well as pollution and various gas levels.

# Importing necessary libraries

In [4]:
import numpy as np
import pandas as pd

import plotly.express as px # all of the graps will be made using plotly express



## Basic statistics

First, let's load the dataset and present some basic statistics.

In [5]:
df = pd.read_csv('/Users/fedor/HSE dsba/Project/AQI and Lat Long of Countries.csv') #loading the dataset

In [6]:
df.shape

(16695, 14)

This dataset contains almost 17000 rows and 14 columns.

In [7]:
df.describe(include='all')

,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category,lat,lng
count,16393,16695,16695.000000,16695,16695.000000,16695,16695.000000,16695,16695.000000,16695,16695.000000,16695,16695.000000,16695.000000
unique,174,14229,NaN,6,NaN,3,NaN,5,NaN,2,NaN,6,NaN,NaN
top,United States of America,Santa Cruz,NaN,Good,NaN,Good,NaN,Good,NaN,Good,NaN,Good,NaN,NaN
freq,3954,17,NaN,7708,NaN,16691,NaN,15529,NaN,16684,NaN,7936,NaN,NaN
mean,NaN,NaN,62.998682,NaN,1.342138,NaN,31.767355,NaN,3.819647,NaN,59.821324,NaN,30.267148,-3.944485
std,NaN,NaN,43.091971,NaN,2.371379,NaN,22.839343,NaN,5.880677,NaN,43.208298,NaN,22.947398,73.037148
min,NaN,NaN,7.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,-54.801900,-171.750000
25%,NaN,NaN,38.500000,NaN,1.000000,NaN,20.000000,NaN,0.000000,NaN,34.000000,NaN,16.515450,-75.180000
50%,NaN,NaN,52.000000,NaN,1.000000,NaN,29.000000,NaN,2.000000,NaN,52.000000,NaN,38.815800,5.643100
75%,NaN,NaN,69.000000,NaN,1.000000,NaN,38.000000,NaN,5.000000,NaN,69.000000,NaN,46.683300,36.275000


- The average AQI values is around 63 points, with the highest being 500 and the lowest being 7.
- The dataset contains 174 various countries, with the most common being the USA, with almost 4000 occurences in this table.
- For all 4 pollutants(CO, O3, NO2 and PM2.5), the most common category is "Good".
- The data also includes the coordinates of the all its cities, which are recorded in the 'Lat' and 'Long' columns.

Let's look at the first 10 rows of the dataset to get a better visual understanding of the data.

In [8]:
df.head(10)

,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category,lat,lng
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate,44.7444,44.2031
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good,-5.2900,-44.4900
2,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good,-11.2958,-41.9869
3,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate,37.1667,15.1833
4,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good,53.0167,20.8833
5,United States of America,Punta Gorda,54,Moderate,1,Good,14,Good,11,Good,54,Moderate,16.1005,-88.8074
6,United States of America,Punta Gorda,54,Moderate,1,Good,14,Good,11,Good,54,Moderate,26.8941,-82.0513
7,Belgium,Puurs,64,Moderate,1,Good,29,Good,7,Good,64,Moderate,51.0761,4.2803
8,Russian Federation,Pyatigorsk,54,Moderate,1,Good,41,Good,1,Good,54,Moderate,44.0500,43.0667
9,China,Qinzhou,68,Moderate,2,Good,68,Moderate,1,Good,58,Moderate,21.9500,108.6167


Then let's check for missing values. They are represented by NaN in this dataset.

In [9]:
df.isnull().sum()

Country               302
City                    0
AQI Value               0
AQI Category            0
CO AQI Value            0
CO AQI Category         0
Ozone AQI Value         0
Ozone AQI Category      0
NO2 AQI Value           0
NO2 AQI Category        0
PM2.5 AQI Value         0
PM2.5 AQI Category      0
lat                     0
lng                     0
dtype: int64

As we can see from the table above, the only missing values are the country names. Since there is no sufficient methond to check for missing values, the best option will be to drop these rows, also taking into account that there are not that many missing values.

Let's also look if there are any dublicate entries of the city name in the dataset.

In [10]:
df.duplicated(subset='City').sum()

2466

It means that there are 2466 dublicates in the dataset, which need to be removed.

In [11]:
df.drop_duplicates(subset='City') # drops dublicates
df.dropna(inplace=True) # drops rows with missing values
df.reset_index(drop=True, inplace=True) # resets the indexes
df.isnull().sum() # check to see that there are no missing values left

Country               0
City                  0
AQI Value             0
AQI Category          0
CO AQI Value          0
CO AQI Category       0
Ozone AQI Value       0
Ozone AQI Category    0
NO2 AQI Value         0
NO2 AQI Category      0
PM2.5 AQI Value       0
PM2.5 AQI Category    0
lat                   0
lng                   0
dtype: int64

The nulls and dublicates have been dropped, and we are now ready to proceed with the graph plotting and further analysis.

# Graph plotting

## Simple graphs plotting

In order to visualise the data better, I will create a choropleth map that will show the average AQI values for each country.

In [12]:
fig = px.choropleth(df, locations='Country', locationmode='country names', color='AQI Value', hover_name='Country', color_continuous_scale='deep')
# fig.update_layout(margin=dict(t=0, l=0, r=0, b=0))

Let's also plot the distribution of each category of AQI. 

In [13]:
fig = px.histogram(df, x='AQI Category', color='AQI Category', title='Distribution of AQI Categories', color_discrete_sequence=['#f1c40f', '#58d68d', '#d62728', '#9467bd', '#ff7f0e', 'black'])
# fig.show()

As we can see, most of the cities in the dataset have either Modeate or Good air quality.

And let's also make a pie chart that shows the distribution of categories of a pollutant, for example PM2.5.

In [14]:
fig = px.pie(df, values='PM2.5 AQI Value', names='AQI Category', title='Distribution of PM2.5 AQI Categories', color_discrete_sequence=['#f1c40f', '#58d68d', '#d62728', '#9467bd', '#ff7f0e', 'black'])
fig.show()

## Complicated graphs plotting

To get a sense of the global air quality, we can plot a histogram of the AQI values.
 The larger the value, the more polluted the air the in the city is.
Values below 50 are considered completely safe, while values above 100 are considered unhealthy.

In [15]:
fig = px.histogram(df, x='AQI Value', color='AQI Category', nbins=300, title='Global Air Quality Index', color_discrete_sequence=['#f1c40f', '#58d68d', '#d62728', '#9467bd', '#ff7f0e', 'black'])
fig.show()

From the histogram, we can see that the AQI values are mostly distributed between 0 and 100, with a peak around 50. This indicates that the majority of the world's cities have relatively good air quality, except for a few cities with extremely high AQI values(27 cities with a value of 500).

Finally, I'll create a sunburst chart that will indicate all 4 pollutant categories for each country. The parenting variable(from which the other categories will expand) will the the Country name, and then will come the AQI Category, CO AQI Category, Ozone AQI Category. 

In [16]:
import plotly.express as px

# Prepare the data for the sunbusrt graph
sunburst_data = df.groupby(['Country', 'AQI Category', 'CO AQI Category', 'Ozone AQI Category', 'NO2 AQI Category', 'PM2.5 AQI Category']).size().reset_index(name='Count')

# Calculate total count for each country
country_counts = sunburst_data.groupby('Country')['Count'].sum().reset_index()

# Create a list of countries whose count is larger than 130
new_country = country_counts[country_counts['Count'] > 130]['Country'].tolist()

# create a new dataframe with only countries with count larger than 130
sunburst_dataf = sunburst_data[sunburst_data['Country'].isin(new_country)]

# Calculate the total count for each pollutant category within each country
sunburst_dataf = sunburst_dataf.groupby(['Country', 'CO AQI Category', 'Ozone AQI Category', 'NO2 AQI Category', 'PM2.5 AQI Category'])['Count'].sum().reset_index()

# Create the sunburst plot
fig = px.sunburst(
    sunburst_dataf,
    path=['Country', 'CO AQI Category', 'Ozone AQI Category', 'NO2 AQI Category', 'PM2.5 AQI Category'],
    values='Count',
    title='Distribution of AQI Values and Pollutants by Country',
    width=700,
    height=700
)

# fig.show()

# A more detailed overview

As as have various pollutants in the table, I belive that we can construct a matrix to look at the correlation between different pollutants and how their changes affect each other.

In [17]:
# i'll create a separate dataframe for all the pollutants.
pol_df = df[['CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']]

# Let's create a correlation matrix
corr_matrix = pol_df.corr().round(2)
corr_matrix

,CO AQI Value,Ozone AQI Value,NO2 AQI Value,PM2.5 AQI Value
CO AQI Value,1.00,0.04,0.40,0.46
Ozone AQI Value,0.04,1.00,-0.25,0.23
NO2 AQI Value,0.40,-0.25,1.00,0.34
PM2.5 AQI Value,0.46,0.23,0.34,1.00


And also let's create a heatmap to better visualize this correlation matrix.

In [18]:
fig = px.imshow(corr_matrix, text_auto=True, title='Correlation Matrix of Pollutants and AQI', color_continuous_scale='haline')
fig.show()

As we can see from the heatmap:
- PM2.5 has quite a strong positive correlation with all other pollutants.
- NO2 has a pretty strong negative correlation with Ozone. This means that when the level of NO2 increases, the level of Ozone will decrease.

# Hypothesis creation and testing

The hypothesis I'm considering is that **The larger the population of the city, the higher is the AQI Value.**

Since I don't have a "Population column in my dataset, I will get this column from an external source.


## Getting a new database with population data 

In [19]:
c_df = pd.read_csv('/Users/fedor/HSE dsba/Project/worldcities.csv')
c_df.rename(columns={'city' : 'City', 'population' : 'Population'}, inplace=True) # renaming the columns for consistency
c_sdf = c_df[['City', 'Population']].copy() # creating a smaller dataframe with only the necessary columns

Let's also clean the subframe from missing values and duplicates.

In [20]:
c_sdf.dropna(inplace=True) # drops rows with missing values
c_sdf.drop_duplicates(subset='City', inplace=True) # drops dublicates
c_sdf.reset_index(drop=True, inplace=True) # resets the indexes

## Merge the two datasets

Let's merge the two datasets on the 'City' column.

In [21]:
dfm = df.merge(c_sdf, on='City', how='left').copy() # merging the two datasets
dfm.drop(3679, inplace=True) # drops american city Delhi as it causes chaos in the dataframe

Let's look at the new dataframe to check if there are any missing values.

In [22]:
dfm.isnull().sum()

Country                0
City                   0
AQI Value              0
AQI Category           0
CO AQI Value           0
CO AQI Category        0
Ozone AQI Value        0
Ozone AQI Category     0
NO2 AQI Value          0
NO2 AQI Category       0
PM2.5 AQI Value        0
PM2.5 AQI Category     0
lat                    0
lng                    0
Population            92
dtype: int64

We can see that not all values have been matched, and now we have to decide if we want to drop these rows or use a different method to fill in the missing values.


In [23]:
missing_population_rows = dfm[dfm['Population'].isnull()]
missing_population_rows.sort_values('AQI Value', ascending=False).head(10)

,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category,lat,lng,Population
13359,South Africa,Nelspruit,237,Very Unhealthy,13,Good,3,Good,7,Good,237,Very Unhealthy,-25.4745,30.9703,NaN
9492,India,Tonk,191,Unhealthy,3,Good,36,Good,7,Good,191,Unhealthy,26.1505,75.7900,NaN
12390,Nigeria,Yenagoa,164,Unhealthy,8,Good,16,Good,6,Good,164,Unhealthy,4.9267,6.2676,NaN
3337,India,Berasia,164,Unhealthy,2,Good,57,Moderate,2,Good,164,Unhealthy,23.6146,77.4645,NaN
14198,India,Vasai,158,Unhealthy,3,Good,28,Good,3,Good,158,Unhealthy,19.4700,72.8000,NaN
11531,Uzbekistan,Buka,152,Unhealthy,1,Good,46,Good,0,Good,152,Unhealthy,-5.4219,154.6728,NaN
2699,China,Zhoucun,123,Unhealthy for Sensitive Groups,2,Good,123,Unhealthy for Sensitive Groups,4,Good,94,Moderate,37.4509,115.4829,NaN
663,China,Matou,111,Unhealthy for Sensitive Groups,2,Good,42,Good,8,Good,111,Unhealthy for Sensitive Groups,39.5503,116.1063,NaN
11218,India,Bandipur,102,Unhealthy for Sensitive Groups,1,Good,48,Good,0,Good,102,Unhealthy for Sensitive Groups,27.9381,84.4069,NaN
366,Benin,Dogbo,101,Unhealthy for Sensitive Groups,3,Good,18,Good,2,Good,101,Unhealthy for Sensitive Groups,6.8167,1.7833,NaN


Since the AQI value in the top 10 cities is pretty high, I think that it will be better to fill in these values rather than drop them.

In [24]:
dfm['Population'] = dfm['Population'].fillna(dfm['Population'].mean()) #filling in missing values with the mean population
dfm['Population'] = dfm['Population'].astype(int) # converting the float values to integers
dfm.isnull().sum() # check to see that there are no missing values left

Country               0
City                  0
AQI Value             0
AQI Category          0
CO AQI Value          0
CO AQI Category       0
Ozone AQI Value       0
Ozone AQI Category    0
NO2 AQI Value         0
NO2 AQI Category      0
PM2.5 AQI Value       0
PM2.5 AQI Category    0
lat                   0
lng                   0
Population            0
dtype: int64

To test the hypothesis, I will create a scatter plot to visualize the relationship between the population and the AQI Value.

In [25]:
fig = px.scatter(dfm, x='Population', y='AQI Value', hover_name='City', title='Population vs AQI Value', color = 'AQI Value', color_continuous_scale='bluered', trendline='ols', trendline_color_override='#1aff1a')
fig.show()

From the scatter plot we can derive that therer are several dots in the top left corner indicating that despite having a relatively small population, their AQI Value is an astonishingly high value of 500. And if we look at the bottom right corner, Tokyo, despite its gargantuan population of 37 mil has a relatively low AQI Value of 79.

In [26]:
fig = px.scatter_geo(dfm,
                     lat='lat',
                     lon='lng',
                     color='AQI Value',
                     size='Population',
                     hover_name='City',
                     title='Population vs AQI Value',
                     labels={'AQI Value': 'AQI Value', 'Population': 'Population'},
                     )

fig.show()

As we can see from the map, the largest dots not always have the highest values of AQI. This could be due to various factors such as population density, geographical location, and the industialisation level of the city, since the pollutants in the table (PM2.5, NO2 and CO all get produced when the heavy industry is working.)

**In conclusion, the hypothesis that the larger the population of the city, the higher is the AQI Value is not supported by the given data. The data does not provide enough evidence to confirm this hypothesis, as there were several occurences when the data didn't match the hypothesis**